# Funnel Analysis & Drop-Off Detection

This notebook reproduces the assignment funnel, quantifies the drop-off at each stage, identifies the biggest bottleneck, and saves the chart plus report in the output folder.

In [ ]:
from collections import OrderedDict
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

stages = OrderedDict([
    ('Sign Up', 10000),
    ('Email Entered', 8000),
    ('Password Created', 6000),
    ('Email Verified', 5000),
    ('Payment Added', 4000),
    ('First Purchase', 2000),
])
revenue_per_customer = 100
stages

In [ ]:
stage_names = list(stages.keys())
stage_values = list(stages.values())

drop_rows = []
for index in range(len(stage_values) - 1):
    users_before = stage_values[index]
    users_after = stage_values[index + 1]
    users_lost = users_before - users_after
    drop_rows.append({
        'from_stage': stage_names[index],
        'to_stage': stage_names[index + 1],
        'users_before': users_before,
        'users_after': users_after,
        'users_lost': users_lost,
        'completion_rate': round((users_after / users_before) * 100, 1),
        'drop_rate': round((users_lost / users_before) * 100, 1),
        'revenue_impact': users_lost * revenue_per_customer,
    })

funnel_df = pd.DataFrame(drop_rows)
impact_df = funnel_df.copy()
impact_df['priority'] = impact_df['revenue_impact'].apply(lambda value: 'HIGH' if value > 100000 else 'MEDIUM')
highest_impact = impact_df.sort_values(by=['revenue_impact', 'drop_rate', 'users_lost'], ascending=[False, False, False]).iloc[0]

display(funnel_df)
display(impact_df.sort_values(by=['revenue_impact', 'drop_rate', 'users_lost'], ascending=[False, False, False]))
highest_impact

In [ ]:
output_dir = Path('../output')
output_dir.mkdir(parents=True, exist_ok=True)

fig, ax = plt.subplots(figsize=(12, 6))
colors = ['#3b82f6', '#10b981', '#f59e0b', '#ef4444', '#8b5cf6', '#ec4899']
ax.bar(list(stages.keys()), list(stages.values()), color=colors)
ax.set_ylabel('Users', fontsize=12)
ax.set_xlabel('Stage', fontsize=12)
ax.set_title('Signup Funnel: Volume by Stage', fontsize=14)
ax.set_ylim(0, max(stages.values()) * 1.15)

for stage, count in stages.items():
    ax.text(stage, count, f'{count:,}', ha='center', va='bottom', fontweight='bold')

plt.xticks(rotation=45, ha='right')
plt.tight_layout()
chart_path = output_dir / 'funnel_chart.png'
plt.savefig(chart_path, dpi=150)
plt.show()
chart_path

In [ ]:
additional_conversions = int(highest_impact['users_lost'] * 0.10)
additional_revenue = additional_conversions * revenue_per_customer

recommendation = f'''FUNNEL OPTIMIZATION PRIORITY

CRITICAL BOTTLENECK:
Stage: {highest_impact['from_stage']} → {highest_impact['to_stage']}
Users Lost: {highest_impact['users_lost']:,.0f}
Completion Rate: {highest_impact['completion_rate']:.1f}%
Drop Rate: {highest_impact['drop_rate']:.1f}%
Revenue Impact: ${highest_impact['revenue_impact']:,.0f}

ROOT CAUSE HYPOTHESES:
- The step may be too complex or too long.
- The step may be unclear, with weak guidance or messaging.
- The step may require too much trust too early in the journey.
- The step may suffer from technical friction or form errors.

RECOMMENDED ACTION:
1. A/B test a simplified version of the step.
2. Reduce the number of fields or required actions.
3. Add clearer copy, progress cues, and reassurance.
4. Track the drop rate before and after the change.

BUSINESS VALUE OF FIXING THE LEAK:
If completion improves by 10%, additional conversions = {additional_conversions:,.0f} and additional revenue = ${additional_revenue:,.0f}.

SUCCESS CRITERIA:
- Reduce drop rate on this step by at least 5 percentage points.
- Improve completion rate by at least 10% relative to baseline.
- Confirm the change with a statistically valid A/B test.
'''

report_path = output_dir / 'funnel_analysis.txt'
report_path.write_text(recommendation, encoding='utf-8')
report_path